# 00 · The Contest — predicting a network's *mean behavior* from its *weights*

**The ARC White-Box Estimation Challenge in one sentence:** you are handed the
weights `W` of a deep ReLU network and must predict, *without running it on many
inputs*, the average value each neuron outputs when the input is Gaussian noise.

This notebook builds that question from scratch and lets you **play with it
interactively**. The one idea to carry through everything:

> Once the weights `W` are fixed, the input distribution is *integrated out*.
> The answer is a **deterministic function of the weights**, `μ = F(W)`.
> A genie with `W` and unlimited compute computes `F(W)` exactly, zero error.
> The sport is computing `F(W)` **cheaply**.

Turn the `SCALE` knob (next cell) from `"tiny"` to `"competition"` and re-run —
every plot below rescales with it.

*(Companion notebooks: `01_the_method` — our learned solution; `02_dynamical_systems`
— why `F` behaves the way it does. See `README.md`.)*

In [1]:
import sys
import numpy as np
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, IntSlider, FloatSlider

sys.path.insert(0, ".")            # so `config.py` (same dir) imports
from config import PRESETS, describe

SCALE = "competition"                    # <-- the one knob: "tiny" | "small" | "medium" | "competition"
cfg = PRESETS[SCALE]
dev = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print(f"SCALE = {SCALE!r}  ({dev})")
print(describe(cfg))

SCALE = 'competition'  (cuda)
width d=256, depth K=32  ->  W in R^2097152, F: R^2097152 -> R^256   (MC M=8192, n_train=50000, channels c=32)


## 1. The network and its forward map `f`

Each layer takes a vector, multiplies by a weight matrix, and applies ReLU
(clip negatives to zero). Stacking `K` layers gives the forward map

$$ f(x;W) \;=\; \mathrm{ReLU}\!\big(W_{K}\,\mathrm{ReLU}(W_{K-1}\cdots \mathrm{ReLU}(W_1 x))\big),
   \qquad x\in\mathbb{R}^{d}. $$

Weights are **He-initialized**, $W_{\ell,ij}\sim\mathcal N(0,\,2/d)$ — the scaling
that keeps activation magnitudes roughly constant with depth (neither exploding
nor vanishing). We stack all layers into $W\in\mathbb{R}^{K\times d\times d}$, so
$\mathrm{flat}(W)\in\mathbb{R}^{Kd^2}$ is the raw input the challenge gives you.

*(This mirrors `../whest_toy.py`; we re-derive it here so the math sits next to the code.)*

In [2]:
def sample_weights(seed, d, K, var=None):
    # W ~ N(0, 2/d) He init; shape (K, d, d).  var overrides the scale if you want.
    var = 2.0 / d if var is None else var
    g = torch.Generator().manual_seed(seed)
    return (torch.randn(K, d, d, generator=g, dtype=torch.float64) * var ** 0.5).float()

def push(W, X):
    # Push a shared batch of inputs X:(n,d) through every network in W:(B,K,d,d).
    # z <- ReLU(W_i z) at each layer, done for all B networks at once.
    # returns final-layer activations, shape (B, n, d).
    z = X.unsqueeze(0).expand(W.shape[0], -1, -1)                 # (B, n, d)
    for i in range(W.shape[1]):
        z = torch.relu(torch.einsum("bne,bfe->bnf", z, W[:, i]))  # apply layer i to all
    return z

d, K = cfg["width"], cfg["depth"]
W1 = sample_weights(0, d, K).to(dev)[None]                        # one network: (1,K,d,d)
x = torch.randn(5, d, device=dev)                                 # 5 example inputs
print("one input -> one output:", push(W1, x)[0, 0].cpu().numpy().round(3))
print("output is nonnegative (post-ReLU):", bool((push(W1, x) >= 0).all()))

one input -> one output: [2.55  0.991 0.    0.42  1.196 0.175 1.642 0.886 0.    0.    0.187 1.666
 0.    0.93  0.282 0.    1.251 0.443 0.    1.659 0.    0.    0.98  0.
 3.745 2.94  0.009 0.    2.211 0.524 1.117 1.228 0.    0.264 0.882 2.049
 0.    0.    1.537 1.266 0.571 0.    0.848 0.    0.    0.    1.013 0.
 2.053 0.    1.827 0.    0.    0.    0.849 1.632 1.758 0.    0.    0.573
 0.    0.    2.165 0.613 0.    0.    0.092 0.989 0.    0.    0.    2.309
 0.    0.    0.    0.119 0.    0.    1.963 1.336 0.    0.    0.    0.
 0.    2.509 0.    0.    1.23  0.    0.    1.845 0.    0.    0.141 0.138
 1.103 1.251 0.733 0.126 0.    2.863 0.    0.    0.    1.986 2.709 0.
 1.907 0.    0.696 0.045 0.    2.288 0.    0.    0.    0.    0.    1.832
 1.819 3.485 1.366 0.    0.    2.972 0.065 1.834 1.464 0.    0.    3.077
 0.    0.    0.    0.    1.194 0.    0.    2.692 0.    0.387 0.    0.
 0.    1.205 0.    1.263 0.    0.    0.    0.617 1.019 0.782 1.532 0.
 0.    0.138 0.    0.377 0.058 0.    0.    0

## 2. The target `F(W)` and its Monte-Carlo estimate

The quantity to predict is the **per-neuron mean activation** under Gaussian input:

$$ F(W) \;=\; \mathbb{E}_{x\sim\mathcal N(0,I_d)}\big[f(x;W)\big]\ \in\ \mathbb{R}^{d}. $$

It's an expectation, so the obvious estimate is a **Monte-Carlo average**: draw
$M$ inputs, push them through, average.

$$ \widehat F_{\mathrm{MC}}(W) \;=\; \frac1M\sum_{m=1}^{M} f(x_m;W),
   \qquad x_m\sim\mathcal N(0,I_d). $$

The key picture: the $M$ per-sample outputs form a **cloud**; `F(W)` is the fixed
point that cloud is scattered around. Scrub the widget: the histogram is the cloud
(one output coordinate), the dashed line is the MC mean estimating `F`.

In [3]:
def mc_samples(W, M, seed=1):
    g = torch.Generator(device=dev).manual_seed(seed)
    X = torch.randn(M, W.shape[-1], generator=g, device=dev)      # (M, d)
    return push(W, X)[0]                                          # (M, d) one network's cloud

def cloud_plot(seed=0, width=d, depth=K, coord=0, log2_M=12):
    W = sample_weights(seed, width, depth).to(dev)[None]
    Z = mc_samples(W, 2 ** log2_M)                                # (M, width)
    vals = Z[:, coord].cpu().numpy()
    fig = go.Figure()
    fig.add_trace(go.Histogram(x=vals, nbinsx=60, name="per-sample output"))
    fig.add_vline(x=vals.mean(), line_color="crimson", line_width=3,
                  annotation_text=f"MC mean = {vals.mean():.3f}")
    fig.update_layout(title=f"Output-coordinate {coord}: the cloud and its mean "
                            f"(seed {seed}, d={width}, K={depth}, M={2**log2_M})",
                      xaxis_title="activation value", yaxis_title="count",
                      template="plotly_white", height=380)
    fig.show()

interact(cloud_plot, seed=IntSlider(0, 0, 30), width=IntSlider(d, 2, 256),
         depth=IntSlider(K, 1, 32), coord=IntSlider(0, 0, 256 - 1),
         log2_M=IntSlider(12, 6, 16));

interactive(children=(IntSlider(value=0, description='seed', max=30), IntSlider(value=256, description='width'…

**Notice** how much probability mass sits *exactly at zero* — a neuron is
silent whenever its pre-activation is negative. The mean is pulled toward zero by
all those dead samples. (At tiny width and large depth you may see the whole cloud
collapse to zero — that's the "dead network" pathology; more in `02_dynamical_systems`.)

### Monte-Carlo error shrinks like `1/√M`

The MC mean is itself random — a different draw of inputs gives a slightly
different number. Its standard error is $\sigma_f/\sqrt{M}$ (the CLT), where
$\sigma_f$ is the spread of the cloud. So to halve the error you need $4\times$ the
samples. Watch the running mean settle as `M` grows:

In [4]:
def convergence_plot(seed=0, width=d, depth=K, coord=0):
    W = sample_weights(seed, width, depth).to(dev)[None]
    Z = mc_samples(W, 2 ** 16)[:, coord].cpu().numpy()           # 65536 samples
    Ms = np.unique(np.round(np.logspace(1, 4.8, 40)).astype(int))
    running = np.array([Z[:m].mean() for m in Ms])
    truth = Z.mean()
    se = Z.std() / np.sqrt(Ms)                                    # sigma_f / sqrt(M)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=Ms, y=running, mode="lines+markers", name="running MC mean"))
    fig.add_trace(go.Scatter(x=Ms, y=truth + se, mode="lines", line=dict(dash="dot", color="gray"),
                             name="±1 std-error band"))
    fig.add_trace(go.Scatter(x=Ms, y=truth - se, mode="lines", line=dict(dash="dot", color="gray"),
                             fill="tonexty", fillcolor="rgba(0,0,0,0.06)", showlegend=False))
    fig.add_hline(y=truth, line_color="crimson", annotation_text="best estimate of F")
    fig.update_layout(title="MC mean converges as 1/√M", xaxis_title="M (samples)",
                      xaxis_type="log", yaxis_title=f"mean of coord {coord}",
                      template="plotly_white", height=380)
    fig.show()

interact(convergence_plot, seed=IntSlider(0, 0, 30), width=IntSlider(d, 2, 64),
         depth=IntSlider(K, 1, 32), coord=IntSlider(0, 0, d - 1));

interactive(children=(IntSlider(value=0, description='seed', max=30), IntSlider(value=64, description='width',…

## 3. The Unscented Transform — a *cheap* estimate

Monte-Carlo is wasteful: it spends samples reproducing the input distribution's
shape. The **unscented transform (UT)** instead uses a small, *deterministic* set
of "sigma points" chosen so their sample mean and covariance **exactly** match the
input $\mathcal N(0,I_d)$:

$$ \xi \in \{\pm\sqrt{d}\,e_i\}_{i=1}^{d}\quad(2d\text{ points}),\qquad
   \tfrac{1}{2d}\sum_\xi \xi = 0,\quad \tfrac{1}{2d}\sum_\xi \xi\xi^\top = I_d. $$

Push those through and average. Because the first two moments are matched exactly,
the estimate is accurate to higher order than random sampling with the same number
of points. We **randomize** it: apply a random rotation $Q$ (which preserves the
moment-matching) and average several rotations to wash out the fixed-axis bias.

$$ \widehat F_{\mathrm{UT}}(W) \;=\; \frac{1}{2dR}\sum_{r=1}^{R}\sum_{i=1}^{d}
   \Big[f(\sqrt d\,Q_r e_i) + f(-\sqrt d\,Q_r e_i)\Big]. $$

This is the family our leaderboard submission lives in. **`2 d R` points total.**

In [5]:
def ut_points(d, rotations, seed=0):
    # randomized UT sigma points for N(0, I_d): +- sqrt(d) * (rotated axes).
    g = torch.Generator(device=dev).manual_seed(seed)
    blocks = []
    for _ in range(rotations):
        A = torch.randn(d, d, generator=g, device=dev)
        Q, _ = torch.linalg.qr(A)                                 # random orthonormal axes
        axes = d ** 0.5 * Q.T                                     # rows: +-sqrt(d) directions
        blocks += [axes, -axes]
    return torch.cat(blocks, 0)                                   # (2*d*rotations, d)

def ut_mean(W, rotations, seed=0):
    S = ut_points(W.shape[-1], rotations, seed)                   # sigma points
    return push(W, S)[0].mean(0)                                  # (d,) equal-weight average

S = ut_points(d, cfg["ut_rotations"])
print(f"UT uses {S.shape[0]} points (= 2 * d={d} * rotations={cfg['ut_rotations']})")
print("sigma-point mean ~0:", float(S.mean(0).abs().max()),
      " cov ~ I:", bool(torch.allclose(S.T @ S / S.shape[0], torch.eye(d, device=dev), atol=1e-4)))

UT uses 6656 points (= 2 * d=256 * rotations=13)
sigma-point mean ~0: 5.587935447692871e-09  cov ~ I: True


## 4. The money experiment — UT vs MC, head to head

Take many random networks. For each, compute a near-exact "truth" (huge-MC), then
compare two cheap estimators **at the same point budget**: the UT, and plain MC.
Plot the histogram of the **final-layer error** `estimate − truth` across networks.

A tighter histogram = a better estimator at that budget. The UT should be
noticeably tighter than MC — that's *why* it's the leaderboard tool. Slide the UT
rotations and the width and watch both histograms move.

In [6]:
def head_to_head(width=d, depth=K, rotations=cfg["ut_rotations"], n_nets=10, truth_M=20000):
    seeds = list(range(n_nets))
    W = torch.stack([sample_weights(s, width, depth) for s in seeds]).to(dev)   # (B,K,d,d)
    n_pts = 2 * width * rotations                                 # UT budget
    # truth: big MC
    g = torch.Generator(device=dev).manual_seed(777)
    Xt = torch.randn(truth_M, width, generator=g, device=dev)
    truth = push(W, Xt).mean(1)                                  # (B, width)
    # UT at n_pts, and MC at the SAME n_pts
    S = ut_points(width, rotations, seed=1)
    ut = push(W, S).mean(1)
    g2 = torch.Generator(device=dev).manual_seed(2)
    Xm = torch.randn(n_pts, width, generator=g2, device=dev)
    mc = push(W, Xm).mean(1)
    ut_err = (ut - truth).flatten().cpu().numpy()                # final-layer errors, all coords
    mc_err = (mc - truth).flatten().cpu().numpy()
    fig = go.Figure()
    fig.add_trace(go.Histogram(x=mc_err, nbinsx=80, name=f"MC ({n_pts} pts)  std={mc_err.std():.4f}",
                               opacity=0.6))
    fig.add_trace(go.Histogram(x=ut_err, nbinsx=80, name=f"UT ({n_pts} pts)  std={ut_err.std():.4f}",
                               opacity=0.6))
    fig.update_layout(barmode="overlay", title=f"Final-layer error vs truth "
                      f"(d={width}, K={depth}, {n_nets} networks)",
                      xaxis_title="estimate - truth", yaxis_title="count",
                      template="plotly_white", height=400)
    fig.show()

interact(head_to_head, width=IntSlider(d, 2, 256), depth=IntSlider(K, 1, 32),
         rotations=IntSlider(cfg["ut_rotations"], 1, 20));

interactive(children=(IntSlider(value=256, description='width', max=256, min=2), IntSlider(value=32, descripti…

## 5. Why *width* is the friend of every estimator

At the competition's width **256**, something helpful happens: **concentration**.
Mean-field theory says that at He criticality every neuron's mean activation
converges to a *universal constant* $\sqrt{1/\pi}\approx 0.564$, and the
network-to-network spread of `F` shrinks. At small width the story is messier:
heavy tails (a few "chaotic" networks with huge activations) and even fully-dead
networks. Sweep the width and watch the distribution of `‖F‖` tighten and its mean
climb toward the dashed line:

In [7]:
def concentration_plot(depth=K, n_nets=300):
    widths = [4, 8, 16, 32, 64]
    fig = go.Figure()
    stats = []
    for w in widths:
        W = torch.stack([sample_weights(s, w, depth) for s in range(n_nets)]).to(dev)
        g = torch.Generator(device=dev).manual_seed(9)
        X = torch.randn(4096, w, generator=g, device=dev)
        F = push(W, X).mean(1)                                   # (n_nets, w)
        per_neuron = F.flatten().cpu().numpy()                   # pool coords (exchangeable)
        fig.add_trace(go.Violin(y=per_neuron, name=f"d={w}", box_visible=True, meanline_visible=True))
        stats.append((w, per_neuron.mean()))
    fig.add_hline(y=(1/np.pi) ** 0.5, line_dash="dash", line_color="crimson",
                  annotation_text="mean-field 0.564")
    fig.update_layout(title=f"Per-neuron mean activation concentrates as width grows (K={depth})",
                      yaxis_title="F entries", template="plotly_white", height=430)
    fig.show()
    print("width -> mean(F):", {w: round(m, 3) for w, m in stats})

interact(concentration_plot, depth=IntSlider(K, 1, 32));

interactive(children=(IntSlider(value=32, description='depth', max=32, min=1), IntSlider(value=300, descriptio…

## 6. The scoring, and why this becomes a pure-accuracy race

The leaderboard metric is (lower is better)

$$ \text{score} \;=\; \frac1M\sum_{m}\, \underbrace{\text{final\_layer\_MSE}_m}_{\text{accuracy}}
   \times \underbrace{\max(0.1,\ C_m/B)}_{\text{compute multiplier}}, $$

with `C_m` the compute you spend and `B` the budget. The crucial quirk: the
multiplier **floors at 0.1** for anything using ≤ 10 % of the budget (~6500 forward
passes at width 256). UT with a few thousand points is far under that floor — so
**compute is effectively free and the whole game is minimizing the final-layer
MSE.** Everyone at the top runs the multiplier pinned at 0.1; ranks differ only by
accuracy. That's why better *quadrature* (more UT rotations, higher-order rules)
climbs the board — and why our research question is really *"how well can you
compute `F(W)`?"*

### Takeaways
- `F(W)` is **deterministic** in the weights; estimating it is a **quadrature**
  problem, and **UT beats MC** at equal budget (§4).
- **Width helps** (concentration, §5); **depth hurts** (correlations compound) —
  explored in `02_dynamical_systems`.
- UT is the strong, *boring* baseline. **Can a model instead LEARN `F` directly
  from the weights `W`?** That is `01_the_method`.